* The original example asked users to set `DB_URI` to a database. This version uses deterministic local data, so neither `DB_URI` nor a database is required.

In [ ]:
import datetime as dt


In [ ]:
import numpy as np
import pyarrow as pa


In [ ]:
import pyarrow.compute as pc

rng = np.random.default_rng(42)


In [ ]:
        # const linear model
n_dates = 6
n_assets = 12


In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
# %%time
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
import calc_flow as cf


In [ ]:
ic = await cf.compute_async(
    panel,
    lambda input_table: input_table.sql("""
        SELECT trade_date, CORR(factor, dx) AS ic
        FROM input
        GROUP BY trade_date
        ORDER BY trade_date
    """),
)
assert ic.num_rows == n_dates


In [ ]:
assert np.isfinite(ic["ic"].to_numpy(zero_copy_only=False)).all()
ic
